<a href="https://colab.research.google.com/github/KejHo/Google-Colab/blob/main/Gemma4_E2B-v_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦥 Gemma 4 Fine-tuning from origin

### 📊 Technické parametry & Statistiky
*   **Model:** Gemma 4 (2B-it)
*   **Hardware:** Google Colab T4 GPU (~15GB VRAM)
*   **Využití VRAM:** ~7.5 GB (při tréninku s 4-bit kvantizací)
*   **Metoda:** LoRA (Rank: 16, Alpha: 16)
*   **Dataset:** FineTome-100k (subset 5k)
*   **Optimalizace:** 2x rychlejší trénink díky Unsloth patchování

### 🔗 Odkazy
*   **GitHub:** [Tvůj repozitář](https://github.com/KejHo/Google-Colab)

---

# 🛠️ Šablona pro trénování Gemma 4 (Unsloth)

Tento notebook slouží jako univerzální šablona. Stačí nastavit parametry a spustit vše najednou.

In [ ]:
# --- KONFIGURACE ---
# Tato sekce definuje klíčové parametry pro fine-tuning modelu, optimalizované pro Google Colab free tier.

# MODEL_NAME: Název základního modelu, který bude fine-tunován.
# 'unsloth/gemma-4-E2B-it' je zvolen jako efektivní 2B model, ideální pro omezené GPU zdroje free tieru.
# Alternativy: Větší modely (např. 7B a více) by vyžadovaly placenou verzi Colabu nebo snížení všech ostatních parametrů.
MODEL_NAME = "unsloth/gemma-4-E2B-it"

# DATASET_NAME: Název datasetu z Hugging Face Hub, který bude použit pro trénink.
# 'mlabonne/FineTome-100k' je vybrán jako ukázkový dataset.
# Pro free tier je klíčové omezit velikost datasetu, proto bude použit jen jeho subset.
DATASET_NAME = "mlabonne/FineTome-100k"

# MAX_STEPS: Maximální počet tréninkových kroků (iterations).
# Hodnota 300 je nastavena pro demonstraci a pro minimalizaci času a rizika pádu na free tieru.
# Vyšší hodnoty (tisíce) by sice zlepšily kvalitu finetuningu, ale výrazně by prodloužily trénink a zvýšily riziko timeoutu/pádu na free tieru.
MAX_STEPS = 300

# LORA_R: Rank LoRA adaptérů. 'r' určuje dimenzionalitu matic použitých v LoRA adaptérech.
# Hodnota 16 je dobrý kompromis pro free tier. Nižší rank (např. 8) by šetřil více VRAM, ale model by se méně adaptoval.
# Vyšší rank (např. 32, 64) by vedl k lepší adaptaci, ale spotřeboval by příliš mnoho VRAM a hrozil by pád na free tieru.
LORA_R = 16

# OUTPUT_DIR: Název adresáře, kam se uloží výstupy tréninku.
# Důležité pro organizaci výstupů a možnost pokračovat v tréninku nebo model deployovat.
OUTPUT_DIR = "gemma_4_lora"

# BATCH_SIZE: Velikost dávky (batch size) pro trénink na jedno GPU.
# Hodnota 2 je zvolena jako velmi malá dávka pro maximální úsporu VRAM na GPU s omezenou pamětí (jako Colab T4).
# Menší dávka může vést k nestabilnějším gradientům, ale je nezbytná pro trénink větších modelů na free tieru.
BATCH_SIZE = 2

# GRADIENT_ACCUMULATION_STEPS: Počet kroků, po kterých se akumulují gradienty před provedením optimalizačního kroku.
# Pomáhá simulovat větší efektivní batch size (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS), když je VRAM omezená.
# Zde efektivní batch size = 2 * 4 = 8. To pomáhá dosáhnout stabilnějšího tréninku i s malou skutečnou dávkou bez nároků na větší VRAM.
# Alternativy: Lze zvýšit pro simulaci ještě větších dávek, ale prodlužuje tréninkový čas.
GRADIENT_ACCUMULATION_STEPS = 4

### Instalace

In [ ]:
%%capture
import os, re, torch, gc

# Tato buňka instaluje všechny potřebné Python balíčky pro fine-tuning.
# '%%capture' na začátku buňky potlačuje výstup instalace, aby byl notebook čistější.

# Podmínka 'COLAB_' kontroluje, zda se kód spouští v prostředí Google Colab.
# Tím se zajistí, že instalace proběhne pouze tam, kde je to potřeba.
if "COLAB_" in "".join(os.environ.keys()):
    # Zjištění verze PyTorche: Důležité pro kompatibilitu s 'xformers'.
    # Používá regulární výraz k extrakci hlavní verze (např. '2.10' z '2.10.0+cu128').
    v = re.match(r'[\u0200-\u02FF\u0300-\u036F\u1E00-\u1EFF\u0400-\u04FF\u0500-\u052F\u2DE0-\u2DFF\uA640-\uA69F\d]{1,}\.[\u0200-\u02FF\u0300-\u036F\u1E00-\u1EFF\u0400-\u04FF\u0500-\u052F\u2DE0-\u2DFF\uA640-\uA69F\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    # Instalace základních knihoven pro práci s daty a Hugging Face ekosystémem:
    # - `sentencepiece`, `protobuf`: Často vyžadované tokenizerem a knihovnou Hugging Face Transformers.
    # - `datasets`: Knihovna pro efektivní práci s datovými sadami.
    # - `huggingface_hub`: Pro interakci s Hugging Face Hub (stahování modelů, datasetů).
    # - `hf_transfer`: Urychluje stahování souborů z Hugging Face Hub.
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

    # Instalace knihoven optimalizovaných pro rychlý fine-tuning LLM:
    # - `unsloth_zoo`, `unsloth`: Hlavní knihovna pro optimalizaci tréninku LLM, slibuje 2x zrychlení.
    # - `bitsandbytes`: Pro 4-bit kvantizaci modelů, což drasticky snižuje spotřebu VRAM a umožňuje trénink větších modelů na slabším hardware.
    # - `accelerate`: Knihovna od Hugging Face pro snadný distribuovaný trénink a optimalizace.
    # - `xformers`: Implementuje optimalizované Attention mechanismy, zrychluje trénink a snižuje spotřebu VRAM.
    # - `peft`: Parameter-Efficient Fine-Tuning, knihovna obsahující implementace jako LoRA.
    # - `trl`: Transformer Reinforcement Learning, knihovna pro trénink LLM (obsahuje `SFTTrainer`).
    # - `triton`: Jazyk a kompilátor pro psaní vysoce optimalizovaných CUDA kernelů, využíván Unslothem.
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

    # Instalace knihovny `transformers` v konkrétní verzi pro zajištění kompatibility.
    # Verze 5.5.0 je doporučená pro Unsloth v době psaní.
    !pip install --no-deps transformers==5.5.0

    # Instalace a upgrade knihoven `torchao` a `timm` pro další optimalizace.
    # `torchao` (Torch Advanced Optimizations) a `timm` (PyTorch Image Models) mohou obsahovat nízkoúrovňové optimalizace.
    !pip install --upgrade torchao timm

In [ ]:
# Tato buňka obsahuje zakomentovanou redundantní instalaci `torchao`.
# Byla přesunuta do předchozí buňky pro centralizovanou instalaci závislostí.
# Ponecháno jako připomínka, že dříve mohla být instalována samostatně.
# !pip install --upgrade torchao # Tato instalace je již zahrnuta v předchozí buňce, je zde redundantní.


In [ ]:
%%capture
# Tato buňka obsahuje zakomentovanou redundantní instalaci `timm`.
# Byla přesunuta do buňky s instalací všech závislostí pro centralizaci.
# Ponecháno s '%%capture' pro minimalizaci výstupu, pokud by byla náhodou spuštěna.
# !pip install --no-deps --upgrade timm # Tato instalace je již zahrnuta v předchozí buňce, je zde redundantní.


### Načtení modelu

In [ ]:
from unsloth import FastModel
import torch

# Tato buňka je klíčová pro načtení předtrénovaného jazykového modelu.

# FastModel.from_pretrained: Metoda z knihovny Unsloth, která načte model a tokenizer.
# Unsloth modely jsou optimalizované pro rychlejší a paměťově úspornější trénink.
model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,     # Název modelu načtený z konfigurace (zde 'unsloth/gemma-4-E2B-it').
                                 # Unsloth automaticky aplikuje optimalizace pro daný typ modelu.
    dtype = None,                # Datový typ (např. torch.float16, torch.bfloat16).
                                 # 'None' nechá Unsloth automaticky vybrat nejlepší datový typ pro dané GPU a optimalizace.
                                 # Pro T4 GPU je `bfloat16` obvykle preferovaný, pokud je podporován, pro lepší numerickou stabilitu.
    max_seq_length = 1024,       # Maximální délka sekvence tokenů, kterou model dokáže zpracovat.
                                 # Delší sekvence umožňují modelu vnímat více kontextu, ale zvyšují spotřebu VRAM a výpočetní nároky.
                                 # Zde 1024 je rozumný kompromis, pro složitější úlohy může být potřeba více.
    load_in_4bit = True,         # Klíčový parametr pro úsporu VRAM. Načte model v 4-bitové kvantizaci.
                                 # To snižuje paměťovou stopu modelu až 4x, což je nezbytné pro trénink velkých modelů na běžných GPU.
                                 # Alternativy: `load_in_8bit=True` (menší úspora), `load_in_4bit=False` (plná přesnost, mnohem více VRAM).
    full_finetuning = False,     # Určuje, zda se bude provádět plné fine-tuning nebo Parameter-Efficient Fine-Tuning (PEFT, např. LoRA).
                                 # 'False' znamená, že budeme používat PEFT (viz další buňka), což šetří VRAM a tréninkový čas.
                                 # 'True' by trénovalo všechny parametry modelu, což je paměťově i výpočetně velmi náročné.
    # token = "TVUJ_HF_TOKEN",  # Volitelný parametr pro přístup k privátním modelům na Hugging Face Hub.
                                 # Pokud model vyžaduje autentizaci, zde se zadává token z Hugging Face.
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

### Přidání LoRA adaptérů

In [ ]:
# Tato buňka aplikuje LoRA (Low-Rank Adaptation) adaptéry na načtený model.
# LoRA je technika Parameter-Efficient Fine-Tuning (PEFT), která dramaticky snižuje počet trénovatelných parametrů.

# FastModel.get_peft_model: Metoda Unsloth pro aplikaci LoRA.
# Vytvoří malé 'adaptérové' matice, které se vkládají do existujících vrstev modelu a trénují se pouze ony.
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Určuje, zda se budou trénovat vrstvy spojené se zpracováním obrazu.
                                       # Zde 'False', protože Gemma je čistě textový model a tato funkcionalita by byla redundantní.
    finetune_language_layers   = True,  # Určuje, zda se budou trénovat jazykové vrstvy (např. embeddingy, vrstvy attention a MLP).
                                       # 'True' je zásadní pro adaptaci modelu na nové jazykové vzorce a fakta.
    finetune_attention_modules = True,  # Specifické trénování mechanismů pozornosti (attention).
                                       # Attention vrstvy jsou klíčové pro porozumění kontextu v LLM.
    finetune_mlp_modules       = True,  # Specifické trénování MLP (Multi-Layer Perceptron) vrstev.
                                       # MLP vrstvy zpracovávají transformace tokenů a hrají důležitou roli v reprezentaci informací.
    r = LORA_R,                  # Rank LoRA adaptérů (z konfigurace, zde 16).
                                 # 'r' určuje dimenzionalitu nových matic, které se učí. Hodnota 16 je zvolena jako dobrý kompromis
                                 # mezi vyjádřovací silou a spotřebou VRAM, což je kritické pro free tier Colab.
    lora_alpha = 16,             # Faktor škálování pro LoRA adaptéry. Obvykle se rovná 'r'.
                                 # Ovlivňuje, jak moc se nové naučené váhy z LoRA adaptérů projevují v původním modelu.
    lora_dropout = 0,            # Dropout rate pro LoRA vrstvy. '0' znamená žádný dropout.
                                 # Dropout je technika regularizace, která pomáhá zabránit přeučení tím, že náhodně "vypíná" neurony během tréninku.
                                 # Pro LoRA adaptéry se často nastavuje na 0 nebo velmi nízkou hodnotu, protože se trénuje jen malá část parametrů.
    bias = "none",               # Typ biasu pro LoRA vrstvy. 'none' je častá a efektivní volba pro úsporu parametrů.
                                 # Alternativy: 'all' (přidá bias pro všechny vrstvy), 'lora_only' (jen pro LoRA vrstvy).
    random_state = 3407,         # Seed pro generátor náhodných čísel. Zajišťuje reprodukovatelnost výsledků tréninku.
)

### Příprava datasetu

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Tato buňka připravuje dataset pro trénink a zajišťuje, že data budou ve správném formátu.

# get_chat_template: Funkce z Unsloth, která konfiguruje tokenizer pro správné formátování konverzací.
# Modely jako Gemma očekávají specifickou strukturu pro promptování (např. <|user|> ... <|model|> ...).
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4", # Specifikuje chat template pro model Gemma 4. To je klíčové pro konzistentní vstup.
)

# load_dataset: Funkce z knihovny `datasets` pro načtení datasetu z Hugging Face Hub.
# Klíčový pro free tier: 'split="train[:5000]"' načte POUZE prvních 5000 příkladů z tréninkového splitu datasetu.
# Tímto se drasticky snižuje paměťová náročnost a doba tréninku, aby se zabránilo pádu Colabu na free tieru.
# Pro plný trénink by se použil celý tréninkový split (`split="train"`), ale to by vyžadovalo více VRAM a času.
dataset = load_dataset(DATASET_NAME, split = "train[:5000]")

# train_test_split: Rozdělení načteného datasetu na tréninkovou a validační část.
# Validační část slouží k monitorování výkonu modelu na neviděných datech během tréninku a pomáhá detekovat přeučení.
# 'test_size = 0.1' znamená, že 10% dat bude použito pro validaci, a zbylých 90% pro trénink.
dataset = dataset.train_test_split(test_size = 0.1)
train_dataset = dataset["train"] # Tréninková část datasetu.
eval_dataset = dataset["test"]   # Validační část datasetu.

In [ ]:
# Tato buňka je duplicitní a načítá a rozděluje dataset znovu.
# Je redundantní, jelikož stejná operace je již provedena v předchozí buňce (6PG-uGJnKTut).
# Tato buňka by měla být odstraněna, aby nedocházelo ke zbytečným operacím a plýtvání pamětí, což je klíčové pro optimalizaci na Colab free tieru.

# Načtení a rozdělení na train/test pro validaci
# dataset = load_dataset(DATASET_NAME, split = "train[:5000]") # Zakomentováno: Redundantní načítání datasetu.
# dataset = dataset.train_test_split(test_size = 0.1) # Zakomentováno: Redundantní rozdělení datasetu.
# train_dataset = dataset["train"] # Zakomentováno: Již definováno.
# eval_dataset = dataset["test"]   # Zakomentováno: Již definováno.

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import standardize_data_formats

# Tato buňka standardizuje formát dat, což je klíčové pro správné fungování Unsloth traineru.

# standardize_data_formats: Funkce z Unsloth, která transformuje dataset do formátu, jenž očekává trainer.
# Zajišťuje, že sloupce jako 'messages' nebo 'conversations' jsou správně zpracovány.
# Důležité pro to, aby `SFTTrainer` mohl správně aplikovat chat template a tokenizaci.
train_dataset = standardize_data_formats(train_dataset)
eval_dataset = standardize_data_formats(eval_dataset)

print("Formáty datasetů byly úspěšně standardizovány.")

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/4500 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Formáty datasetů byly úspěšně standardizovány.


In [ ]:
# Tato buňka slouží k rychlé vizuální kontrole formátu dat po všech úpravách.

# Zobrazení ukázky 101. prvku (s indexem 100, protože indexování začíná od 0) z tréninkové části datasetu.
# Výstup by měl ukázat strukturu konverzace, která bude předána modelu.
# Pomáhá ověřit, že data byla načtena a formátována správně a že jsou připravena pro další krok.
train_dataset[100]

{'conversations': [{'role': 'user',
   'content': "Can you explain the purpose of __str__ and __repr__ in Python? I understand that __str__ returns the string representation of an object, but I'm not sure why and when I would need to use it. Additionally, I'm uncertain about the usage and application of __repr__."},
  {'role': 'assistant',
   'content': "The `__str__` and `__repr__` methods in Python are used for object string representations. \n\nThe `__str__` method in Python represents the class objects as a string – it can be used for classes. The `__str__` method should be defined in a way that is easy to read. This method is also used as a debugging tool when the need arises. \n\nHere's an example of how `__str__` might be used:\n\n```python\nclass Person:\n    def __init__(self, person_name, person_age):\n        self.name = person_name\n        self.age = person_age \n\n    def __str__(self):\n        return f'Person(name={self.name}, age={self.age})'\n\np = Person('John', 30)\

### Formátování datasetu

**OPRAVA:** Původní verze používala `processor` místo `tokenizer`, což způsobovalo `NameError`. Opraveno níže.

In [ ]:
def formatting_prompts_func(examples):
    # Tato funkce je definována pro formátování vstupních promptů do struktury, kterou model očekává.
    # Je aplikována na dataset pomocí metody `.map()`.

    # Extrahujeme konverzace z příkladů datasetu.
    convos = examples["conversations"]
    texts = [
        # Aplikujeme chat template na každou konverzaci.
        # 'tokenizer.apply_chat_template' je klíčová pro transformaci strukturované konverzace
        # (např. [{role: user, content: "hi"}, {role: assistant, content: "hello"}])
        # na řetězec, který model rozumí (např. '<|user|> hi <|model|> hello').
        tokenizer.apply_chat_template(
            convo,
            tokenize = False, # 'False' znamená, že výstupem je string (řetězec), ne tokeny (celá čísla).
                                # Tokenizace proběhne později v SFTTraineru.
            add_generation_prompt = False # 'False' zabraňuje přidání speciálního tokenu pro generování na konci.
                                         # To je důležité pro trénink, kde chceme, aby model doplňoval odpověď, ne začínal novou generaci.
        ).removeprefix('<bos>') # `.removeprefix('<bos>')` odstraní počáteční token `<bos>` (Begin-Of-Sentence), který je často přidán automaticky.
                                # Někdy se tento token na začátku konverzace nechce pro finetuning, neboť model by měl začínat "přirozeněji".
        for convo in convos
    ]
    # Vrátí slovník s klíčem 'text', který obsahuje naformátované konverzace jako řetězce.
    # Tento klíč 'text' bude použit `SFTTrainer`em jako `dataset_text_field`.
    return {"text": texts}

try:
    # Mapování formátovací funkce na obě části datasetu (tréninkový a validační).
    # 'batched = True' umožňuje zpracovávat příklady v dávkách pro efektivitu.
    # Tato operace transformuje dataset, aby obsahoval nový sloupec 'text' s naformátovanými konverzacemi.
    train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
    eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)
    print("Dataset úspěšně naformátován.")
except NameError as e:
    # Ošetření chyby: Pokud 'tokenizer' není definován (např. nebyla spuštěna buňka s načtením modelu).
    # Poskytuje uživateli srozumitelnou zprávu, jak chybu opravit.
    print(f"Chyba: {e}. Spusťte prosím buňky v sekci 'Načtení modelu' a 'Příprava datasetu'.")

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset úspěšně naformátován.


In [ ]:
# Tato buňka slouží ke konečné kontrole naformátovaného textu, který bude sloužit jako vstup pro trénink.

# Zobrazení 101. příkladu z tréninkového datasetu po aplikaci `formatting_prompts_func`.
# Měl by ukázat, jak přesně bude vypadat vstupní řetězec pro model, včetně všech speciálních tokenů z chat template.
# Kontrola je důležitá pro ověření, že struktura konverzace je správně převedena a odpovídá očekávání modelu.
train_dataset[100]['text']

"<|turn>user\nCan you explain the purpose of __str__ and __repr__ in Python? I understand that __str__ returns the string representation of an object, but I'm not sure why and when I would need to use it. Additionally, I'm uncertain about the usage and application of __repr__.<turn|>\n<|turn>model\nThe `__str__` and `__repr__` methods in Python are used for object string representations. \n\nThe `__str__` method in Python represents the class objects as a string – it can be used for classes. The `__str__` method should be defined in a way that is easy to read. This method is also used as a debugging tool when the need arises. \n\nHere's an example of how `__str__` might be used:\n\n```python\nclass Person:\n    def __init__(self, person_name, person_age):\n        self.name = person_name\n        self.age = person_age \n\n    def __str__(self):\n        return f'Person(name={self.name}, age={self.age})'\n\np = Person('John', 30)\nprint(p)\n```\n\nWhen you print the object `p` of class 

### Nastavení tréneru

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# Tato buňka konfiguruje a inicializuje SFTTrainer, což je vysoce optimalizovaný tréner pro fine-tuning LLM.

try:
    # SFTTrainer (Supervised Fine-Tuning Trainer): Nástroj z knihovny `trl` pro efektivní trénink LLM.
    # Zjednodušuje proces tréninku tím, že se stará o detaily jako je tokenizace, LoRA integrace a nastavení tréninkové smyčky.
    trainer = SFTTrainer(
        model = model,                   # Model, který budeme trénovat (zde s již aplikovanými LoRA adaptéry).
        tokenizer = tokenizer,           # Tokenizer pro zpracování textu, musí být stejný jako ten, který byl použit pro formátování dat.
        train_dataset = train_dataset,   # Tréninkový dataset, který jsme připravili v předchozích krocích.
        eval_dataset = eval_dataset,     # Validační dataset pro vyhodnocování během tréninku. Důležité pro monitorování pokroku.
        args = SFTConfig(               # SFTConfig: Třída pro nastavení hyperparametrů tréninku.
            dataset_text_field = "text", # Klíč ve slovníku datasetu, který obsahuje naformátovaný text pro trénink.
            per_device_train_batch_size = BATCH_SIZE, # Skutečná velikost dávky na jedno GPU (z konfigurace).
                                                      # Tato hodnota je speciálně upravená pro maximální úsporu VRAM na free tieru Colab.
            gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS, # Akumulace gradientů (z konfigurace) pro simulaci větší dávky.
                                                                       # Pomáhá efektivně obcházet omezení paměti free tieru a dosáhnout stabilnějšího tréninku.
            warmup_steps = 10,           # Počet kroků, během kterých se learning rate (rychlost učení) postupně zvyšuje na svou maximální hodnotu.
                                         # Pomáhá stabilizovat trénink na začátku.
            max_steps = MAX_STEPS,       # Maximální celkový počet tréninkových kroků (z konfigurace).
                                         # Hodnota je nastavena nízká pro free tier, aby se minimalizoval čas a riziko pádu.
            learning_rate = 2e-4,        # Počáteční rychlost učení. Klíčový hyperparametr, který ovlivňuje rychlost a stabilitu tréninku.
                                         # Běžné hodnoty pro LoRA se pohybují od 1e-5 do 5e-4.
            fp16 = not torch.cuda.is_bf16_supported(), # Použití 16-bitové plovoucí desetinné čárky (FP16), pokud BF16 není podporováno.
                                                      # FP16 šetří paměť a zrychluje výpočty, ale může být méně stabilní než BF16.
            bf16 = torch.cuda.is_bf16_supported(),     # Použití BFloat16, pokud je podporováno GPU (např. T4, V100, A100).
                                                      # BF16 nabízí lepší numerickou stabilitu než FP16 a zároveň úsporu paměti oproti FP32.
            logging_steps = 1,           # Jak často (po kolika krocích) se mají logovat tréninkové metriky (ztráta, learning rate).
                                         # '1' znamená logování každého kroku pro detailní sledování.
            optim = "adamw_8bit",        # Optimalizátor: AdamW optimalizátor s 8-bitovou přesností.
                                         # `adamw_8bit` z `bitsandbytes` je vysoce paměťově efektivní verze AdamW, ideální pro LLM fine-tuning na omezeném hardware.
            weight_decay = 0.01,         # Míra regularizace (L2) pro váhy modelu.
                                         # Pomáhá zabránit přeučení tím, že penalizuje velké váhy.
            lr_scheduler_type = "cosine",# Typ plánovače learning rate. 'cosine' postupně snižuje learning rate podle kosinové křivky.
                                         # Pomáhá modelu usadit se v optimálním bodě.
                                         # Alternativy: 'linear', 'constant', 'polynomial'.
            seed = 3407,                 # Seed pro generátor náhodných čísel pro reprodukovatelnost experimentů.
            eval_strategy = "steps",     # Strategie vyhodnocování: 'steps' znamená vyhodnocování po určitém počtu tréninkových kroků.
                                         # Alternativa: 'epoch' (po každé epoše).
            eval_steps = 50,             # Jak často se má provádět vyhodnocení na validačním datasetu (každých 50 kroků).
                                         # Poskytuje pravidelný přehled o výkonu modelu.
            save_strategy = "steps",     # Strategie ukládání modelu: 'steps' znamená ukládání po určitém počtu tréninkových kroků.
            save_steps = 50,             # Jak často se má ukládat kontrolní bod (checkpoint) modelu (každých 50 kroků).
                                         # Důležité pro obnovení tréninku po přerušení nebo pro výběr nejlepšího modelu.
            load_best_model_at_end = True, # Na konci tréninku načte nejlepší uložený model (na základě validační metriky, obvykle ztráty).
                                           # Zajišťuje, že finální model je ten, který měl nejlepší validační výkon.
            output_dir = OUTPUT_DIR,     # Adresář pro ukládání výstupů tréninku (z konfigurace).
        ),
        # Callbacks: Funkce, které se spouštějí v různých fázích tréninku.
        callbacks = [EarlyStoppingCallback(early_stopping_patience=3)], # EarlyStoppingCallback: Callback pro předčasné zastavení tréninku.
                                                                       # 'early_stopping_patience=3' znamená, že trénink se zastaví, pokud se validační ztráta
                                                                       # nezlepší po dobu 3 po sobě jdoucích evaluačních kroků. Pomáhá předcházet přeučení a šetří čas,
                                                                       # což je klíčové v omezeném prostředí Colab free tieru.
    )
    print("Trainer úspěšně inicializován.")
except Exception as e:
    print(f"Chyba při inicializaci: {e}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer úspěšně inicializován.


In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Tato buňka aplikuje specializovanou funkci pro maskování vstupů.
# Cílem je zajistit, aby se model učil generovat pouze odpovědi asistenta, nikoli uživatelské prompry.

# train_on_responses_only: Funkce z Unsloth, která modifikuje tréninkový dataset.
# Změní 'labels' (cílové tokeny pro učení) tak, že tokeny odpovídající uživatelským promptům jsou nastaveny na -100.
# Hodnota -100 je speciální maskovací hodnota v Hugging Face Transformer knihovnách, která říká tréneru, aby tyto tokeny ignoroval při výpočtu ztráty.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",  # Předpona, která identifikuje začátek uživatelských instrukcí v chat template.
                                         # Všechny tokeny před touto předponou a mezi ní a `response_part` budou maskovány.
    response_part = "<|turn>model\n", # Předpona, která identifikuje začátek odpovědi modelu.
                                       # Tokeny po této předponě (až do další `instruction_part` nebo konce sekvence) budou trénovány.
)

Map (num_proc=5):   0%|          | 0/4500 [00:00<?, ? examples/s]

Filter (num_proc=5):   0%|          | 0/4500 [00:00<?, ? examples/s]

Unsloth: Removed 14 out of 4500 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Map (num_proc=5):   0%|          | 0/500 [00:00<?, ? examples/s]

Filter (num_proc=5):   0%|          | 0/500 [00:00<?, ? examples/s]

Unsloth: Removed 1 out of 500 samples from eval_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [ ]:
# Tato buňka ověřuje, jak vypadá vstup do modelu (input_ids) po maskování.

# Zobrazuje dekódované vstupní ID pro 101. příklad (index 100) z tréninkového datasetu.
# Zde by měl být vidět celý dialog (jak uživatelský prompt, tak i odpověď modelu).
# Důvod: `input_ids` představují, co model vidí jako vstup. Maskování se aplikuje na `labels` pro výpočet ztráty, ne na samotný vstup.
tokenizer.decode(trainer.train_dataset[100]['input_ids'])

"<|turn>user\nCan you explain the purpose of __str__ and __repr__ in Python? I understand that __str__ returns the string representation of an object, but I'm not sure why and when I would need to use it. Additionally, I'm uncertain about the usage and application of __repr__.<turn|>\n<|turn>model\nThe `__str__` and `__repr__` methods in Python are used for object string representations. \n\nThe `__str__` method in Python represents the class objects as a string – it can be used for classes. The `__str__` method should be defined in a way that is easy to read. This method is also used as a debugging tool when the need arises. \n\nHere's an example of how `__str__` might be used:\n\n```python\nclass Person:\n    def __init__(self, person_name, person_age):\n        self.name = person_name\n        self.age = person_age \n\n    def __str__(self):\n        return f'Person(name={self.name}, age={self.age})'\n\np = Person('John', 30)\nprint(p)\n```\n\nWhen you print the object `p` of class 

In [ ]:
# Tato buňka ověřuje efekt maskování na cílových labelech (labels).

# Dekóduje `labels` pro 101. příklad z tréninkového datasetu.
# Hodnoty `-100` (které byly použity pro maskování tokenů, které nemají přispívat ke ztrátě) jsou nahrazeny `tokenizer.pad_token_id`.
# Následně se `tokenizer.pad_token` (což je obvykle prázdný řetězec nebo speciální znak) nahradí prázdnou mezerou.
# Výsledek: Měl bys vidět pouze text odpovědi modelu, zatímco uživatelský prompt by měl být nahrazen prázdnými mezerami/paddingem.
# To demonstruje, že model se učí generovat pouze odpovědi a ignoruje uživatelské vstupy při výpočtu ztráty.
tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in trainer.train_dataset[100]['labels']
]).replace(tokenizer.pad_token, ' ')

"                                                                     The `__str__` and `__repr__` methods in Python are used for object string representations. \n\nThe `__str__` method in Python represents the class objects as a string – it can be used for classes. The `__str__` method should be defined in a way that is easy to read. This method is also used as a debugging tool when the need arises. \n\nHere's an example of how `__str__` might be used:\n\n```python\nclass Person:\n    def __init__(self, person_name, person_age):\n        self.name = person_name\n        self.age = person_age \n\n    def __str__(self):\n        return f'Person(name={self.name}, age={self.age})'\n\np = Person('John', 30)\nprint(p)\n```\n\nWhen you print the object `p` of class `Person`, it will output: `Person(name=John, age=30)`, which is the string representation of the object.\n\nThe `__repr__` method returns a string that describes how to recreate the object. It is mainly used for debugging and deve

### Trénink

In [ ]:
# Tato buňka slouží k monitorování využití GPU paměti před tréninkem a k samotnému spuštění tréninku.

# Statistiky GPU před tréninkem:
# `torch.cuda.get_device_properties(0)`: Získá informace o prvním dostupném GPU (index 0).
# `gpu_stats.total_memory`: Celková dostupná paměť GPU.
# `torch.cuda.max_memory_reserved()`: Maximální množství paměti, které bylo v danou chvíli alokováno (rezervováno) pro CUDA operace.
# Tyto výpočty pomáhají určit, kolik paměti je k dispozici a kolik se využije, což je klíčové pro optimalizaci nastavení pro Colab Free tier.
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Spuštění tréninku modelu:
# `trainer.train()`: Tato funkce spouští hlavní tréninkovou smyčku s konfigurací nastavenou v `SFTTrainer`u.
# Během této fáze probíhá forward pass, výpočet ztráty, backward pass, akumulace gradientů a aktualizace vah LoRA adaptérů.
# Výstupem je `trainer_stats`, který obsahuje souhrn tréninkových metrik.
trainer_stats = trainer.train()

GPU = Tesla T4. Max memory = 14.563 GB.
7.52 GB of memory reserved.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,486 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,3.838960,1.597949
100,1.988834,1.604017
150,1.527354,1.471908
200,1.886696,1.397863
250,1.928603,1.422306
300,1.643291,1.411880


Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/checkpoint-300/tokenizer_config.json.


In [ ]:
# Tato buňka provádí statistiky po tréninku a uvolňuje GPU paměť pro další operace.

# Statistiky po tréninku:
# `used_memory`: Aktuální maximální rezervovaná paměť GPU po skončení tréninku.
# `used_memory_for_lora`: Vypočítá čistou paměť, kterou spotřebovaly LoRA adaptéry a přidružené struktury (rozdíl mezi `used_memory` a `start_gpu_memory`).
# `used_percentage`: Procentuální využití celkové GPU paměti.
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"Trénink trval: {round(trainer_stats.metrics['train_runtime']/60, 2)} minut.")
print(f"Využití GPU: {used_percentage}% ({used_memory} GB)")

# Vyčištění paměti pro následnou inferenci:
# `gc.collect()`: Spustí garbage collector Pythonu, který uvolní nepoužívané objekty z paměti.
# `torch.cuda.empty_cache()`: Vyprázdní mezipaměť GPU paměti.
# Tyto kroky jsou důležité pro uvolnění co nejvíce VRAM, aby bylo možné provádět inferenci nebo další operace bez chyb nedostatku paměti.
gc.collect()
torch.cuda.empty_cache()

Trénink trval: 54.59 minut.
Využití GPU: 93.895% (13.674 GB)


### Inference (testování modelu)

In [ ]:
from transformers import TextStreamer

# Tato buňka demonstruje, jak použít natrénovaný model pro generování textu (inference).

try:
    # Příprava zprávy pro inference (testování) modelu.
    # Formát konverzace musí odpovídat šabloně 'gemma-4', stejně jako u tréninku.
    # Důležité: 'content' je list slovníků, i když je to jen text, protože Gemma podporuje multimodální vstupy.
    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": "Pokračuj v posloupnosti: 1, 1, 2, 3, 5, 8,"}] # Ukázkový prompt pro model.
    }]

    # Aplikace chat template a tokenizace zprávy. Výstup je připraven pro model.
    # `tokenizer.apply_chat_template` převede strukturu zpráv na jeden řetězec tokenů, který model přijme.
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True, # 'True' přidá speciální token na konec, který signalizuje modelu, že má začít generovat.
                                     # To je standardní postup pro inference, kde model má pokračovat v konverzaci.
        return_tensors = "pt",        # Vrátí PyTorch tenzory, které jsou vyžadovány modelem.
        tokenize = True,              # Tokenizuje výsledný řetězec do ID tokenů.
        return_dict = True,           # Vrátí slovník obsahující `input_ids` a `attention_mask`.
    ).to("cuda") # Přesune vstupní tenzory na GPU, kde je model načten, pro rychlejší výpočty.

    # Generování odpovědi modelem.
    # `model.generate` je metoda pro generování textu.
    _ = model.generate(
        **inputs,                    # Rozbalí slovník `inputs` (input_ids, attention_mask) do argumentů funkce.
        max_new_tokens = 64,         # Maximální počet nově generovaných tokenů (slov/částí slov).
                                     # Určuje délku generované odpovědi.
        temperature = 1.0,           # Teplota pro generování (ovlivňuje náhodnost).
                                     # Vyšší teplota (např. 1.0-1.5) vede k kreativnějším a rozmanitějším odpovědím.
                                     # Nižší teplota (např. 0.7-0.9) vede k předvídatelnějším a faktičtějším odpovědím.
        top_p = 0.95,                # Top-p sampling (nucleus sampling): Filtruje tokeny s nízkou kumulativní pravděpodobností.
                                     # Zajišťuje rozmanitost generace, ale zabraňuje generování nesmyslů.
        top_k = 64,                  # Top-k sampling: Zvažuje pouze `k` nejpravděpodobnějších tokenů při každém kroku generování.
                                     # Také pomáhá kontrolovat náhodnost generace.
        streamer = TextStreamer(tokenizer, skip_prompt = True), # Streamer: Umožňuje postupné zobrazování generovaného textu, jako by model "psal" v reálném čase.
                                                               # `skip_prompt = True` znamená, že uživatelský prompt se nebude opakovat ve výstupu streamerem.
    )
except NameError:
    print("Chyba: Model nebo Tokenizer není načten v paměti.") # Ošetření chyby, pokud by model nebyl načten.


Pokračuješ posloupností **Fibonacciho posloupnosti**.

Další členy posloupnosti jsou:

13, 21, 34, 55, 89,<turn|>


### Uložení modelu

In [ ]:
# Tato buňka ukládá natrénované LoRA adaptéry a konfiguraci tokenizeru lokálně.

# Uložení LoRA adaptérů:
# `model.save_pretrained("gemma_4_lora")`: Uloží pouze váhy LoRA adaptérů (ne celý původní model) do zadaného adresáře.
# To je velmi efektivní z hlediska velikosti souborů, protože LoRA adaptéry jsou malé.
# Z uložených adaptérů lze pak model znovu načíst a použít pro inferenci nebo další trénink.
# model.save_pretrained("gemma_4_lora")

# Uložení konfigurace tokenizeru:
# `tokenizer.save_pretrained("gemma_4_lora")`: Uloží konfiguraci tokenizeru (vocab, speciální tokeny atd.) do stejného adresáře.
# Je nezbytné uložit i tokenizer, aby bylo možné model správně znovu načíst a používat pro tokenizaci vstupů a dekódování výstupů.
# tokenizer.save_pretrained("gemma_4_lora")
# print("LoRA adaptéry uloženy do složky 'gemma_4_lora'")
print("Ukládání LoRA adaptérů lokálně je zakomentováno.")

Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/tokenizer_config.json.


LoRA adaptéry uloženy do složky 'gemma_4_lora'


### Uložení modelu na Google Drive

In [ ]:
import shutil, os
from google.colab import drive

# Tato buňka zajišťuje perzistentní uložení trénovaného modelu na Google Drive.
# Lokální soubory v Colab instanci se po jejím vypnutí ztratí, proto je uložení na Drive klíčové.

try:
    # Připojení Google Drive k prostředí Colab.
    # `drive.mount('/content/drive', force_remount=True)`: Připojí váš Google Drive k adresářové struktuře Colabu.
    # `force_remount=True` zajistí, že se Drive připojí i v případě, že již byl připojen dříve, což je užitečné pro znovuspuštění notebooku.
    # drive.mount('/content/drive', force_remount=True)

    # Definice cesty na Google Drive, kam se model uloží.
    # `MyDrive` je výchozí kořenový adresář vašeho Drive.
    # drive_path = "/content/drive/MyDrive/gemma_4_lora"

    # Kontrola a odstranění existujícího adresáře:
    # `os.path.exists(drive_path)`: Zjistí, zda adresář již existuje.
    # `shutil.rmtree(drive_path)`: Pokud adresář existuje, smaže ho i s celým obsahem.
    # To zajišťuje, že se vždy ukládá nová verze modelu, aniž by se mísila se starými soubory.
    # if os.path.exists(drive_path):
    #     shutil.rmtree(drive_path)

    # Zkopírování lokálně uložených LoRA adaptérů do Google Drive.
    # `shutil.copytree("gemma_4_lora", drive_path)`: Zkopíruje celý adresář 'gemma_4_lora' (který obsahuje model a tokenizer) na zadané místo na Drive.
    # shutil.copytree("gemma_4_lora", drive_path)
    # print(f"✅ Model úspěšně uložen na Google Drive: {drive_path}")
    print("Ukládání LoRA adaptérů na Google Drive je zakomentováno.")
except Exception as e:
    print(f"❌ Chyba při ukládání na Drive: {e}") # Ošetření případných chyb při operacích s Drive.

Mounted at /content/drive
✅ Model úspěšně uložen na Google Drive: /content/drive/MyDrive/gemma_4_lora


### Sloučení modelu a uložení do GGUF formátu pro Ollama Studio

Pro použití v Ollama Studiu je často potřeba mít model ve formátu GGUF. Následující kroky sloučí natrénované LoRA adaptéry s původním modelem a uloží výsledný model do GGUF formátu.

In [ ]:
# Uvolnění GPU paměti pro potenciální sloučení modelu
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastLanguageModel

# Načtení základního modelu a LoRA adaptérů
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = OUTPUT_DIR, # Načte LoRA adaptéry z naší výstupní složky
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

# Sloučení LoRA adaptérů s původním modelem
# Toto vytvoří plný model, který již nepotřebuje LoRA adaptéry samostatně
model.save_pretrained_gguf(
    f"{OUTPUT_DIR}/gemma_4_lora_merged",
    tokenizer = tokenizer,
    # Konfigurace kvantizace pro GGUF. 'q4_k_m' je dobrý kompromis mezi velikostí a výkonem.
    # Další možnosti: 'q4_0', 'q4_1', 'q5_0', 'q5_1', 'q8_0', 'f16'
    quantization_method = "q4_k_m"
)

print(f"✅ Sloučený GGUF model uložen do složky '{OUTPUT_DIR}/gemma_4_lora_merged'")

In [ ]:
import shutil, os
from google.colab import drive

# Tato buňka zajišťuje perzistentní uložení GGUF modelu na Google Drive.

try:
    # Připojení Google Drive k prostředí Colab.
    drive.mount('/content/drive', force_remount=True)

    # Definice cesty na Google Drive, kam se GGUF model uloží.
    gguf_drive_path = f"/content/drive/MyDrive/{OUTPUT_DIR}/gemma_4_lora_merged"
    local_gguf_path = f"{OUTPUT_DIR}/gemma_4_lora_merged"

    # Kontrola a odstranění existujícího adresáře:
    if os.path.exists(gguf_drive_path):
        shutil.rmtree(gguf_drive_path)

    # Zkopírování lokálně uloženého GGUF modelu do Google Drive.
    shutil.copytree(local_gguf_path, gguf_drive_path)
    print(f"✅ GGUF model úspěšně uložen na Google Drive: {gguf_drive_path}")
except Exception as e:
    print(f"❌ Chyba při ukládání GGUF modelu na Drive: {e}")